# PHA Quickstart

Get started with the **Personal Health Insights Agent Team (PHA)** in just a few lines of code.

## What is PHA?

PHA is a multi-agent system that analyzes your health data and provides personalized insights by combining:
- **Data analysis** (trends, patterns, population comparisons)
- **Medical interpretation** (what your metrics mean)
- **Health coaching** (actionable recommendations)

## Requirements

```bash
pip install google-genai pandas openai anthropic
# Optional for full Domain Expert support:
pip install git+https://github.com/google-deepmind/onetwo
```

## Quick Setup

In [ ]:
import os
import sys

# --- Configuration ---
# Set your API key and provider here
API_KEY = "API_KEY"
# Provider options: "gemini", "openai", "anthropic"
PROVIDER = "PROVIDER"
# Optional: Tavily API key for Domain Expert Agent (search)
TAVILY_API_KEY = "TAVILY_API_KEY"

In [ ]:
import pandas as pd
import glob
from pha.agents import (
    DataScienceAgent,
    DomainExpertAgent,
    HealthCoachAgent,
    MultiAgentOrchestrator,
    is_react_available,
)

# Load your health data (using sample data here)
summary_df = pd.read_csv('../data/sample/summary.csv')
activities_df = pd.read_csv('../data/sample/activities.csv')
profile_df = pd.read_csv('../data/sample/profile.csv')
population_df = pd.read_csv('../data/sample/population_percentiles.csv')

print(f"Loaded {len(summary_df)} days of health data")
print(f"ReAct available: {is_react_available()}")


In [ ]:
# Initialize the Data Science Agent
ds_agent = DataScienceAgent()
ds_agent.configure(api_key=API_KEY, provider=PROVIDER)  # Explicit config
ds_agent.load_dataframes({
    'summary': summary_df,
    'activities': activities_df,
    'profile': profile_df,
    'population': population_df,
})
print("✓ Data Science Agent ready")

# Initialize the Domain Expert Agent
de_agent = None
if is_react_available():
    de_agent = DomainExpertAgent(
        search_backend='tavily',
        tavily_api_key=TAVILY_API_KEY,
    )
    exemplar_files = glob.glob('../few_shots/*.ipynb')
    de_agent.get_agent(
        api_key=API_KEY,
        provider=PROVIDER,
        exemplar_files=exemplar_files,
    )
    print(f"✓ Domain Expert Agent ready ({len(exemplar_files)} exemplars)")
else:
    print("⚠ Domain Expert Agent unavailable (install onetwo for full support)")

# Initialize the Health Coach
coach = HealthCoachAgent(simple_mode=True)
coach.configure(api_key=API_KEY, provider=PROVIDER)  # Explicit config
print("✓ Health Coach ready")

# Create the orchestrator with all agents
pha = MultiAgentOrchestrator()
pha.configure(api_key=API_KEY, provider=PROVIDER)  # Explicit config
pha.set_agents(
    data_science_agent=ds_agent,
    domain_expert_agent=de_agent,
    health_coach_agent=coach,
)

print("\n✓ PHA ready!")


## Ask Questions About Your Health

In [ ]:
# Ask a health question
response = pha.respond("How has my sleep been lately?")
print(response)

In [ ]:
# Ask follow-up questions
response = pha.respond("What about my activity levels?")
print(response)

In [ ]:
# Get personalized recommendations
response = pha.respond("What should I focus on to improve my health?")
print(response)

## Using Individual Agents

You can also use each agent independently:

In [ ]:
# Data Science Agent - for data analysis
analysis = ds_agent.query("What's my average daily step count?")
print("Data Science Agent:")
print(analysis)

In [ ]:
# Health Coach - for conversational guidance
coach.reset_conversation()
response = coach.respond("I want to improve my energy levels")
print("Health Coach:")
print(response)

In [ ]:
# Domain Expert - for health interpretation and context
if de_agent:
    response = de_agent.call_agent("What does an HbA1c of 5.8% mean for my health?")
    print("Domain Expert Agent:")
    print(response)
else:
    print("Domain Expert not available (requires onetwo)")


## Next Steps

- See `02_data_science_agent.ipynb` for detailed data analysis
- See `03_domain_expert_agent.ipynb` for health interpretation
- See `04_health_coach_agent.ipynb` for conversational coaching
- See `05_full_pipeline.ipynb` for the complete orchestrator demo

## Loading Your Own Data

```python
# Load your own health data
my_summary = pd.read_csv('path/to/your/summary.csv')
my_activities = pd.read_csv('path/to/your/activities.csv')

ds_agent.load_dataframes({
    'summary': my_summary,
    'activities': my_activities,
})
```

Expected columns:
- **summary**: datetime, steps, sleep_minutes, resting_heart_rate, heart_rate_variability, etc.
- **activities**: start_time, activity_name, duration, calories, steps, etc.